# Phase 1 — Load, Explore & Clean
**Dataset:** Ames Housing Dataset  
**Goal:** Load the data, understand what each column means, and clean it up so we end up with a clean DataFrame ready for analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

---
## Task 1 — Load the Data
We use `pd.read_csv()` to load the dataset and print the first 5 rows to get a quick look at what we're working with.

In [ ]:
df = pd.read_csv("AmesHousing.csv")
df.head()

---
## Task 2 — Check the Shape
`.shape` tells us how many rows and columns are in the dataset.

In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

---
## Task 3 — Inspect & Fix Data Types
We use `.info()` to check all column types. Some columns are stored as the wrong type:

- **`MS SubClass`** is stored as `int64` but it's actually a category code (e.g., 20 = 1-story house built after 1946). It should be `category`.
- **`Bsmt Full Bath`, `Bsmt Half Bath`, `Garage Cars`** are stored as `float64` but they represent whole-number counts. They should be `Int64` (which allows missing values).
- **`Year Built`, `Year Remod/Add`, `Yr Sold`** represent years and are better stored as `datetime` so we can do time-based math later.

In [ ]:
df.info()

In [ ]:
# Fix 1: MS SubClass is a category code, not a real number
df['MS SubClass'] = df['MS SubClass'].astype('category')

# Fix 2: Bath and garage counts should be integers (Int64 supports missing values)
for col in ['Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Cars']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Fix 3: Year columns should be datetime
for col in ['Year Built', 'Year Remod/Add', 'Yr Sold']:
    df[col] = pd.to_datetime(df[col], format='%Y', errors='coerce')

# Fix 4: All text (object) columns → category to save memory
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')

print("Data types fixed!")
df.dtypes.value_counts()

---
## Task 4 — Find & Handle Missing Values
We check which columns have the most missing values. Then we decide what to do:

| Column | Missing | Decision | Reason |
|--------|---------|----------|---------|
| Pool QC | ~99% | **Drop** | Almost entirely missing — not useful |
| Misc Feature | ~96% | **Drop** | Same reason |
| Alley | ~93% | **Drop** | Most houses have no alley — not informative |
| Fence | ~80% | **Drop** | Too many missing to reliably fill |
| Categorical columns | small % | **Fill with mode** | Use the most common value |
| Numeric columns | small % | **Fill with median** | Median is better than mean when data is skewed |

In [ ]:
# Show the top 10 columns with the most missing values
missing = df.isnull().sum().sort_values(ascending=False)
print("Top 10 columns with most missing values:")
print(missing[missing > 0].head(10))

In [ ]:
# Drop columns with too many missing values (more than 50% missing)
cols_to_drop = ['Pool QC', 'Misc Feature', 'Alley', 'Fence']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"Dropped columns: {cols_to_drop}")

# Fill categorical columns with the mode (most frequent value)
for col in df.select_dtypes(include='category').columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Fill numeric columns with the median (robust to outliers)
for col in df.select_dtypes(include=['int64', 'float64', 'Int64']).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Fill datetime columns with the mode
for col in df.select_dtypes(include='datetime64[ns]').columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Missing values remaining: {df.isnull().sum().sum()}")

---
## Task 5 — Handle Duplicates
We check if any rows are exact duplicates and remove them if found.

In [ ]:
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

df = df.drop_duplicates()
print(f"Rows after removing duplicates: {df.shape[0]}")

---
## Task 6 — Spot & Handle Outliers
We use the **IQR method** to find houses with unusually high prices.
Then we cap extreme values at the **99th percentile** so they don't skew our analysis.

In [ ]:
import matplotlib.pyplot as plt

# Step 1: Use IQR to SPOT outliers
# IQR = the range of the middle 50% of prices
Q1 = df['SalePrice'].quantile(0.25)
Q3 = df['SalePrice'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR  # standard outlier threshold

print(f"Q1: ${Q1:,.0f}  |  Q3: ${Q3:,.0f}  |  IQR: ${IQR:,.0f}")
print(f"Outlier threshold (Q3 + 1.5×IQR): ${upper_fence:,.0f}")
print(f"Outliers found: {(df['SalePrice'] > upper_fence).sum()} houses")

# Step 2: Cap at the 99th percentile (as required)
cap_99 = df['SalePrice'].quantile(0.99)
print(f"\n99th percentile cap value: ${cap_99:,.0f}")

In [ ]:
# Cap SalePrice at the 99th percentile
df['SalePrice'] = df['SalePrice'].clip(upper=cap_99)

print(f"Max SalePrice AFTER capping: ${df['SalePrice'].max():,.0f}")

# Histogram after capping — red line shows where we capped
plt.figure(figsize=(8, 4))
plt.hist(df['SalePrice'], bins=40, color='steelblue', edgecolor='black')
plt.axvline(cap_99, color='red', linestyle='--', label=f'99th percentile cap: ${cap_99:,.0f}')
plt.title('SalePrice Distribution — AFTER Capping at 99th Percentile')
plt.xlabel('Price ($)')
plt.ylabel('Number of Houses')
plt.legend()
plt.tight_layout()
plt.show()

---
## Task 7 — The `clean_data()` Function
We wrap all the steps above into a single reusable function. This way we can clean any new dataset with one line of code.

In [ ]:
import pandas as pd

def clean_data(filepath):

    # Step 1 & 2: Load & check shape
    df = pd.read_csv(filepath)
    print(f"Loaded data: {df.shape[0]} rows, {df.shape[1]} columns")

    # Step 3: Fix data types
    df['MS SubClass'] = df['MS SubClass'].astype('category')

    for col in ['Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Cars']:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    for col in ['Year Built', 'Year Remod/Add', 'Yr Sold']:
        df[col] = pd.to_datetime(df[col], format='%Y', errors='coerce')

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')

    # Step 4: Handle missing values
    cols_to_drop = ['Pool QC', 'Misc Feature', 'Alley', 'Fence']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    for col in df.select_dtypes(include='category').columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0])

    for col in df.select_dtypes(include=['int64', 'float64', 'Int64']).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())

    for col in df.select_dtypes(include='datetime64[ns]').columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0])

    # Step 5: Remove duplicates
    before = df.shape[0]
    df = df.drop_duplicates()
    print(f"Removed {before - df.shape[0]} duplicate rows")

    # Step 6: Spot outliers with IQR, then cap at 99th percentile
    Q1 = df['SalePrice'].quantile(0.25)
    Q3 = df['SalePrice'].quantile(0.75)
    IQR = Q3 - Q1
    upper_fence = Q3 + 1.5 * IQR
    print(f"Outliers spotted (IQR method): {(df['SalePrice'] > upper_fence).sum()} houses")

    cap_99 = df['SalePrice'].quantile(0.99)
    df['SalePrice'] = df['SalePrice'].clip(upper=cap_99)
    print(f"SalePrice capped at 99th percentile: ${cap_99:,.0f}")

    print("Cleaning complete!")
    return df


df_clean = clean_data("AmesHousing.csv")
df_clean.head()

---
## Task 8 — Final Checks
Before moving to the next phase, we run 3 checks to make sure our data is clean and ready.

In [ ]:
# Check 1: No missing values in the target column (SalePrice)
assert df_clean['SalePrice'].isnull().sum() == 0, "SalePrice has missing values!"


# Check 2: All sale prices are positive (a price of 0 or negative makes no sense)
assert (df_clean['SalePrice'] > 0).all(), "Some SalePrice values are 0 or negative!"


# Check 3: We have the correct number of columns after dropping 4
expected_cols = 82 - 4 + 0  # original 82 minus 4 dropped (Pool QC, Misc Feature, Alley, Fence)
assert df_clean.shape[1] == expected_cols, f"Expected {expected_cols} columns, got {df_clean.shape[1]}!"



---
## Task 9 — Save the Cleaned Data
We save the cleaned DataFrame so notebook 02 can load it directly without repeating the cleaning steps.
We use pickle (not CSV) because pickle keeps all column types exactly as they are (dates stay as dates, categories stay as categories).

In [ ]:
import os

# Create the folder if it does not exist
os.makedirs("data/cleaned", exist_ok=True)

# Save the cleaned DataFrame as a pickle file so all column types are preserved
df_clean.to_pickle("data/cleaned/ames_cleaned.pkl")

print("Cleaned data saved to: data/cleaned/ames_cleaned.pkl")
print(f"Rows: {df_clean.shape[0]}, Columns: {df_clean.shape[1]}")